# Per-2-Hour-Epoch Patient-Speech-Only Poisson GLM Encoding Orchestrator

Combines the 2-hour-epoch structure of `word_level_duration_cv_filtered_speech_per_epoch.ipynb`
with the patient-speech filtering of `word_level_duration_cv_patient_speech.ipynb`.

Words must pass **both** criteria:
1. **Quality-filtered** (`{patient}_word_df_filtered.csv`)
2. **Patient-speaking** (`{patient}_transcripts_annotated.csv` — LR-ASD marks segment as patient speech)

Epoch blocks: 09-11, 11-13, 13-15, 15-17, 17-19, 19-21, 21-23 **Central local time**.  
One SLURM job is submitted per *(patient, date, epoch)* triplet.  
Outputs land in `{vad_root}/{patient}/encoding/word_level_duration_cv_patient_speech_per_epoch/{date}/{epoch}/`.

### 1. Imports

In [1]:
from pathlib import Path
import subprocess

import dill as pickle
import numpy as np
import pandas as pd

### 2. Configuration

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────────────────
PROJ_DIR    = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247')
VAD_ROOT    = PROJ_DIR / 'vad_new'
WORKER_PATH = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!'
                   '/standard_encoding_analysis/poisson_glm_worker.py')
PYTHON_BIN  = '/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3'

PATIENTS       = None   # None => scan all Y* patient folders
FORCE_RESUBMIT = False

# ── Epoch settings ────────────────────────────────────────────────────────────────────────
# 2-hour blocks in Central local time (inclusive start, exclusive end)
LOCAL_TZ    = 'America/Chicago'
HOUR_BLOCKS = [(9, 11), (11, 13), (13, 15), (15, 17), (17, 19), (19, 21), (21, 23)]
MIN_WORDS_PER_EPOCH = 200   # silently skip epochs below this threshold

# ── Model settings ────────────────────────────────────────────────────────────────────────
SPIKE_OFFSET_IDX = 0
GPT2_LAYER       = -1
N_PCA            = 100
OUTER_SPLITS     = 5
INNER_SPLITS     = 5
N_ALPHAS         = 30
ALPHA_LOW        = -3.0
ALPHA_HIGH       = 3.0
N_SHUFFLES       = 50

# ── Compute mode ─────────────────────────────────────────────────────────────────────────
USE_GPU = True

# ── SLURM ────────────────────────────────────────────────────────────────────────────────
CONDA_ACTIVATE = 'source /scratch/tahaismail424/.bash_profile && conda activate speech_247_env'
if USE_GPU:
    PARTITION = 'Universal'
    GRES      = 'gpu:1'
    CPUS      = 8
    MEM       = '64G'
    TIME      = '24:00:00'
else:
    PARTITION = 'Universal'
    GRES      = ''
    CPUS      = 16
    MEM       = '32G'
    TIME      = '48:00:00'

BASE_RUN_NAME      = 'word_level_duration_cv_patient_speech_per_epoch'
GLOBAL_LOGS_DIR    = VAD_ROOT / f'{BASE_RUN_NAME}_slurm_logs'
GLOBAL_SCRIPTS_DIR = VAD_ROOT / f'{BASE_RUN_NAME}_slurm_scripts'
GLOBAL_LOGS_DIR.mkdir(parents=True, exist_ok=True)
GLOBAL_SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

print('vad root:   ', VAD_ROOT)
print('worker:     ', WORKER_PATH)
print('logs dir:   ', GLOBAL_LOGS_DIR)
print(f'compute:     {"GPU" if USE_GPU else "CPU"} | partition={PARTITION}')
print(f'local tz:    {LOCAL_TZ}')
print(f'hour blocks: {HOUR_BLOCKS}')

vad root:    /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new
worker:      /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_encoding_analysis/poisson_glm_worker.py
logs dir:    /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/word_level_duration_cv_patient_speech_per_epoch_slurm_logs
compute:     GPU | partition=Universal
local tz:    America/Chicago
hour blocks: [(9, 11), (11, 13), (13, 15), (15, 17), (17, 19), (19, 21), (21, 23)]


### 3. Compute & Save Per-Epoch Patient-Speech Word Indices Per Patient

In [3]:
def epoch_str(start: int, end: int) -> str:
    return f'{start:02d}-{end:02d}'


def compute_epoch_indices(patient: str) -> 'dict[tuple, Path]':
    """
    Load quality-filtered word df, intersect with patient-speaking segments,
    assign each word to a 2-hour epoch block, and save one .npy index file
    per (date, epoch) pair.  Returns {(date_str, epoch_str): path}.
    """
    patient_root  = VAD_ROOT / patient
    filtered_csv  = patient_root / f'{patient}_word_df_filtered.csv'
    annotated_csv = patient_root / f'{patient}_transcripts_annotated.csv'

    if not filtered_csv.exists():
        print(f'  [{patient}] missing {filtered_csv.name}')
        return {}
    if not annotated_csv.exists():
        print(f'  [{patient}] missing {annotated_csv.name} — run add_video_diarization_df.ipynb first')
        return {}

    df           = pd.read_csv(filtered_csv, usecols=['original_word_idx', 'utc_word_start'])
    annotated_df = pd.read_csv(annotated_csv, index_col=0)

    patient_speaking = annotated_df['patient_speaking'].astype(bool)
    df['patient_speaking'] = patient_speaking.reindex(df['original_word_idx']).values
    df = df[df['patient_speaking'] == True].copy()

    if df.empty:
        print(f'  [{patient}] no patient-speaking words after quality filter')
        return {}

    df['utc_word_start'] = pd.to_datetime(df['utc_word_start'], utc=True)
    df['local_dt'] = df['utc_word_start'].dt.tz_convert(LOCAL_TZ)
    df['hour']     = df['local_dt'].dt.hour
    df['date_str'] = df['local_dt'].dt.strftime('%Y-%m-%d')

    df['epoch'] = None
    for start, end in HOUR_BLOCKS:
        mask = (df['hour'] >= start) & (df['hour'] < end)
        df.loc[mask, 'epoch'] = epoch_str(start, end)
    df = df[df['epoch'].notna()].copy()

    epoch_paths = {}
    for (date_str, ep), grp in df.groupby(['date_str', 'epoch']):
        if len(grp) < MIN_WORDS_PER_EPOCH:
            continue  # silently skip
        idx_dir  = patient_root / 'encoding' / BASE_RUN_NAME / date_str / ep
        idx_dir.mkdir(parents=True, exist_ok=True)
        idx_path = idx_dir / f'{patient}_{date_str}_{ep}_word_idx.npy'
        word_idx = grp['original_word_idx'].values.astype(np.int64)
        np.save(idx_path, word_idx)
        epoch_paths[(date_str, ep)] = idx_path
        print(f'  [{patient}] {date_str} {ep}: {len(word_idx):,} patient-speech words → {idx_path.name}')

    return epoch_paths


# Discover patients
if PATIENTS is None:
    all_patients = sorted([p.name for p in VAD_ROOT.iterdir() if p.is_dir() and p.name.startswith('Y')])
else:
    all_patients = PATIENTS

print(f'Computing per-epoch patient-speech word indices for {len(all_patients)} patients...\n')
patient_epoch_idx_paths = {}   # {patient: {(date_str, epoch_str): Path}}
for patient in all_patients:
    ep_paths = compute_epoch_indices(patient)
    if ep_paths:
        patient_epoch_idx_paths[patient] = ep_paths

total_jobs = sum(len(v) for v in patient_epoch_idx_paths.values())
print(f'\nTotal (patient, date, epoch) triplets ready: {total_jobs}')

Computing per-epoch patient-speech word indices for 21 patients...

  [YEY] missing YEY_transcripts_annotated.csv — run add_video_diarization_df.ipynb first
  [YEZ] missing YEZ_transcripts_annotated.csv — run add_video_diarization_df.ipynb first
  [YFA] missing YFA_transcripts_annotated.csv — run add_video_diarization_df.ipynb first
  [YFB] missing YFB_transcripts_annotated.csv — run add_video_diarization_df.ipynb first
  [YFC] 2024-07-20 17-19: 274 patient-speech words → YFC_2024-07-20_17-19_word_idx.npy
  [YFC] 2024-07-20 19-21: 207 patient-speech words → YFC_2024-07-20_19-21_word_idx.npy
  [YFC] 2024-07-21 11-13: 401 patient-speech words → YFC_2024-07-21_11-13_word_idx.npy
  [YFC] 2024-07-21 13-15: 425 patient-speech words → YFC_2024-07-21_13-15_word_idx.npy
  [YFC] 2024-07-21 15-17: 389 patient-speech words → YFC_2024-07-21_15-17_word_idx.npy
  [YFC] 2024-07-21 21-23: 492 patient-speech words → YFC_2024-07-21_21-23_word_idx.npy
  [YFC] 2024-07-22 11-13: 659 patient-speech words → Y

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

### 4. Resolve Input Paths & Build Status Table

In [ ]:
def first_existing(paths):
    for path in paths:
        if path is not None and Path(path).exists():
            return Path(path)
    return None


def resolve_patient_base_paths(patient: str) -> dict:
    patient_root = VAD_ROOT / patient
    embeddings_path = first_existing([
        patient_root / 'embeddings' / f'{patient}_gpt2_embeddings.npy',
        patient_root / 'all_convo_recording' / 'all_words_filtered_all_layers_gpt2.npy',
    ])
    counts_path = first_existing([
        patient_root / 'neural_embeddings' / 'word_spike_counts_offsets_all.npy',
        patient_root / 'all_convo_recording' / 'word_spike_counts_offsets_all.npy',
    ])
    durations_path = first_existing([
        patient_root / 'neural_embeddings' / 'word_durs.npy',
        patient_root / 'all_convo_recording' / 'word_durs.npy',
    ])
    return {
        'patient_root': patient_root,
        'embeddings_path': embeddings_path,
        'counts_path': counts_path,
        'durations_path': durations_path,
    }


def build_status_df() -> pd.DataFrame:
    rows = []
    for patient in all_patients:
        base = resolve_patient_base_paths(patient)
        ep_idx_map = patient_epoch_idx_paths.get(patient, {})
        for (date_str, ep), idx_path in ep_idx_map.items():
            out_dir      = base['patient_root'] / 'encoding' / BASE_RUN_NAME / date_str / ep
            result_pkl   = out_dir / f'{patient}_encoding_results_cv.pkl'
            result_tar   = out_dir / f'{patient}_encoding_models_cv.tar'
            meta_json    = out_dir / f'{patient}_meta.json'
            success_path = out_dir / f'{patient}_SUCCESS'
            error_path   = out_dir / f'{patient}_error.txt'

            has_emb  = base['embeddings_path'] is not None
            has_cnt  = base['counts_path'] is not None
            has_dur  = base['durations_path'] is not None
            ready    = has_emb and has_cnt and has_dur
            success  = success_path.exists()

            rows.append({
                'patient':         patient,
                'date':            date_str,
                'epoch':           ep,
                'patient_root':    base['patient_root'],
                'embeddings_path': base['embeddings_path'],
                'counts_path':     base['counts_path'],
                'durations_path':  base['durations_path'],
                'word_idx_path':   idx_path,
                'out_dir':         out_dir,
                'result_pkl':      result_pkl,
                'result_tar':      result_tar,
                'meta_json':       meta_json,
                'success_path':    success_path,
                'error_path':      error_path,
                'has_embeddings':  has_emb,
                'has_counts':      has_cnt,
                'has_durations':   has_dur,
                'ready_inputs':    ready,
                'has_success':     success,
                'has_error':       error_path.exists(),
                'has_result_pkl':  result_pkl.exists(),
                'needs_submit':    ready and (FORCE_RESUBMIT or not success),
            })
    return pd.DataFrame(rows).sort_values(['patient', 'date', 'epoch']).reset_index(drop=True)


status_df = build_status_df()
print(f'(patient, date, epoch) triplets total: {len(status_df)}')
print(f'ready inputs:                          {int(status_df["ready_inputs"].sum())}')
print(f'completed:                             {int(status_df["has_success"].sum())}')
print(f'errors:                                {int(status_df["has_error"].sum())}')
print(f'needs submit:                          {int(status_df["needs_submit"].sum())}')
status_df[['patient', 'date', 'epoch', 'ready_inputs', 'has_success', 'has_error', 'needs_submit']]

### 5. Submit SLURM Jobs

In [ ]:
submitted = []
failed    = []

for _, row in status_df.iterrows():
    patient  = row['patient']
    date_str = row['date']
    ep       = row['epoch']
    if not row['needs_submit']:
        continue

    row['out_dir'].mkdir(parents=True, exist_ok=True)
    reset_line = (
        f"rm -f {row['success_path']} {row['error_path']}" if FORCE_RESUBMIT else 'true'
    )

    cmd = (
        f'{PYTHON_BIN} {WORKER_PATH} {patient} {VAD_ROOT} {row["out_dir"]} '
        f'--spike-offset-idx {SPIKE_OFFSET_IDX} '
        f'--gpt2-layer {GPT2_LAYER} '
        f'--n-pca {N_PCA} '
        f'--outer-splits {OUTER_SPLITS} '
        f'--inner-splits {INNER_SPLITS} '
        f'--n-alphas {N_ALPHAS} '
        f'--alpha-low {ALPHA_LOW} '
        f'--alpha-high {ALPHA_HIGH} '
        f'--n-shuffles {N_SHUFFLES} '
        f'--embeddings-path {row["embeddings_path"]} '
        f'--counts-path {row["counts_path"]} '
        f'--durations-path {row["durations_path"]} '
        f'--word-idx-path {row["word_idx_path"]}'
    )

    job_name   = f'glm_pse_{patient}_{date_str}_{ep}'
    gres_lines = [f'#SBATCH --gres={GRES}'] if GRES else []
    lines = [
        '#!/bin/bash',
        f'#SBATCH --job-name={job_name}',
        f'#SBATCH --partition={PARTITION}',
        *gres_lines,
        f'#SBATCH --cpus-per-task={CPUS}',
        f'#SBATCH --qos=big_batch_tier',
        f'#SBATCH --exclude=guppy-1',
        f'#SBATCH --mem={MEM}',
        f'#SBATCH --time={TIME}',
        f'#SBATCH --output={GLOBAL_LOGS_DIR}/{patient}_{date_str}_{ep}_%j.out',
        f'#SBATCH --error={GLOBAL_LOGS_DIR}/{patient}_{date_str}_{ep}_%j.err',
        '',
        'set -eo pipefail',
        '',
        CONDA_ACTIVATE,
        '',
        'echo "HOSTNAME: $(hostname)"',
        'echo "START: $(date)"',
        f'echo "PATIENT: {patient}"',
        f'echo "DATE: {date_str}"',
        f'echo "EPOCH: {ep}"',
        '',
        reset_line,
        '',
        cmd,
        '',
        'echo "END: $(date)"',
    ]

    sbatch_path = GLOBAL_SCRIPTS_DIR / f'{patient}_{date_str}_{ep}.sbatch'
    sbatch_path.write_text('\n'.join(lines))

    try:
        res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
        submitted.append((patient, date_str, ep, res.stdout.strip()))
        print(f'submitted: {patient} {date_str} {ep} -> {res.stdout.strip()}')
    except subprocess.CalledProcessError as exc:
        failed.append((patient, date_str, ep, exc.stderr.strip()))
        print(f'FAILED SUBMISSION: {patient} {date_str} {ep}\n{exc.stderr}')

print(f'submitted={len(submitted)}, failed={len(failed)}')

### 6. Check Status

In [ ]:
status_df = build_status_df()
print(f'completed: {int(status_df["has_success"].sum())} / {len(status_df)}')
status_df[['patient', 'date', 'epoch', 'ready_inputs', 'has_success', 'has_error', 'has_result_pkl']]

### 7. Inspect Errors

In [ ]:
error_rows = status_df[status_df['has_error']].copy()
print(f'(patient, date, epoch) triplets with error files: {len(error_rows)}')
for _, row in error_rows.head(10).iterrows():
    print('=' * 100)
    print(row['patient'], row['date'], row['epoch'])
    print(row['error_path'].read_text()[:4000])

### 8. Load & Summarise Results

In [ ]:
records = []

for _, row in status_df.iterrows():
    if not row['has_result_pkl']:
        continue
    with open(row['result_pkl'], 'rb') as f:
        df = pickle.load(f)
    if 'is_summary' not in df.columns:
        continue
    summary_df = df[df['is_summary'] == True].copy()
    if summary_df.empty:
        continue
    summary_df['patient'] = row['patient']
    summary_df['date']    = row['date']
    summary_df['epoch']   = row['epoch']
    records.append(summary_df)

if records:
    all_summary = pd.concat(records, ignore_index=True)
    print('loaded (patient, date, epoch) triplets:', len(records))
    print('summary rows:', len(all_summary))
    display(all_summary.head())

    epoch_level = all_summary.groupby(['patient', 'date', 'epoch']).agg(
        n_neurons=('neuron_idx', 'nunique'),
        pseudo_r2_mean=('pseudo_r2_mean', 'mean'),
        pearson_corr_mean=('pearson_corr_mean', 'mean'),
        spearman_corr_mean=('spearman_corr_mean', 'mean'),
    ).reset_index()
    display(epoch_level.sort_values(['patient', 'date', 'epoch']))
else:
    print('No completed results found yet.')